In [1]:
import dionysus as d
import numpy as np
import sys

# 1. Wczytujemy zbiór (np. N=300, żeby skrypt nie trwał wiecznie)
try:
    points = np.loadtxt("torus.csv", delimiter=",")
except OSError:
    print("Wygeneruj najpierw torus.csv dla N=300")
    sys.exit()

print("Budowa jawnego kompleksu...")
f = d.fill_rips(points, 3, 2.0)

# 2. Liczymy wagę wejściową (Sparsity)
# Ile łącznie elementów (wierzchołków) definiuje wszystkie sympleksy w pamięci?
complex_elements = sum(len(simplex) for simplex in f)
print(f"Liczba fizycznych elementów w macierzy brzegów (Wejście): {complex_elements}")

# 3. Odpalamy kohomologię (wymuszamy keep_cocycles=True, żeby je zbadać)
print("\nRedukcja kohomologiczna (czekaj, RAM puchnie)...")
m = d.cohomology_persistence(f, prime=2, keep_cocycles=True)

# 4. Liczymy wagę wyjściową (Density / Fill-in)
# Iterujemy po aktywnych kokyklach i zliczamy, z ilu sympleksów się składają
cocycle_elements = 0
for col in m:
    cocycle_elements += len(col.cocycle)

print(f"Liczba elementów w aktywnych kokyklach (Baza Z_perp): {cocycle_elements}")
print(f"Współczynnik gęstnienia (Fill-in ratio): {cocycle_elements / complex_elements:.2f}x")

# 5. Estymacja twardego RAM-u dla samych kokykli (w C++ każdy indeks to 64-bitowy int / 8 bajtów)
ram_bytes = cocycle_elements * 8
print(f"Czysty rozmiar wektorów indeksów w C++: {ram_bytes / (1024*1024):.2f} MB")

Budowa jawnego kompleksu...
Liczba fizycznych elementów w macierzy brzegów (Wejście): 61013010

Redukcja kohomologiczna (czekaj, RAM puchnie)...
Liczba elementów w aktywnych kokyklach (Baza Z_perp): 13165876
Współczynnik gęstnienia (Fill-in ratio): 0.22x
Czysty rozmiar wektorów indeksów w C++: 100.45 MB


In [2]:
import dionysus as d
import numpy as np
import subprocess
import matplotlib.pyplot as plt
import os

# Parametry testu
point_counts = list(range(100, 750, 50))
max_dim = 2
max_eps = 2.0

# Listy na wyniki
hom_elements_history = []
coh_elements_history = []
complex_elements_history = []

print("Rozpoczynam zliczanie elementów matematycznych (Fill-in Benchmark)...\n")

for N in point_counts:
    print(f"=== Zliczanie dla N = {N} punktów ===")
    
    # 1. Generacja Torusa z kodu C++
    with open("torus.csv", "w") as f:
        subprocess.run(["./torus", str(N), "3"], stdout=f)
        
    points = np.loadtxt("torus.csv", delimiter=",")
    
    # 2. Budowa jawnego kompleksu
    f_complex = d.fill_rips(points, max_dim + 1, max_eps)
    
    # Rozmiar samego kompleksu (dla perspektywy)
    complex_size = sum(len(simplex) for simplex in f_complex)
    complex_elements_history.append(complex_size)
    
    # 3. Test Homologii (Cykle / Redukcja kolumnowa)
    m_hom = d.homology_persistence(f_complex)
    hom_count = sum(len(col) for col in m_hom)
    hom_elements_history.append(hom_count)
    
    # 4. Test Kohomologii (Kokykle / Redukcja wierszowa)
    m_coh = d.cohomology_persistence(f_complex, prime=2, keep_cocycles=True)
    coh_count = sum(len(col.cocycle) for col in m_coh)
    coh_elements_history.append(coh_count)
    
    print(f" -> Homologia:  {hom_count:,} elementów")
    print(f" -> Kohomologia:{coh_count:,} elementów")
    print(f" -> Stosunek:   {coh_count/max(1, hom_count):.2f}x więcej\n")

# ==========================================
# GENEROWANIE WYKRESU DLA PRACY DYPLOMOWEJ
# ==========================================
print("Generowanie wykresu 'benchmark_fill_in.png'...")
os.makedirs("plots", exist_ok=True)

plt.figure(figsize=(10, 6))

# Plotujemy liczbę elementów (Oś Y może być logarytmiczna, ale liniowa pokaże brutalność fill-in)
plt.plot(point_counts, coh_elements_history, marker='^', label='Dionysus Cohomology (Kokykle Z^⊥)', color='blue', linewidth=2)
plt.plot(point_counts, hom_elements_history, marker='s', label='Dionysus Homology (Cykle)', color='red', linewidth=2)

plt.title("Zjawisko Fill-in: Liczba elementów śledzonych podczas redukcji")
plt.xlabel("Liczba punktów w chmurze (3D Torus)")
plt.ylabel("Łączna liczba śledzonych indeksów (Elementów)")

# Dodajemy siatkę i legendę
plt.grid(True, which="both", ls="--")
plt.legend(loc="upper left")

# Formatowanie osi Y, żeby liczby miały separatory tysięcy (np. 10,000,000)
ax = plt.gca()
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))

plt.tight_layout()
plt.savefig("plots/benchmark_fill_in.png", dpi=300)
plt.close()

print("Gotowe! Wykres zapisany w 'plots/benchmark_fill_in.png'.")

Rozpoczynam zliczanie elementów matematycznych (Fill-in Benchmark)...

=== Zliczanie dla N = 100 punktów ===
 -> Homologia:  12,249 elementów
 -> Kohomologia:2,383 elementów
 -> Stosunek:   0.19x więcej

=== Zliczanie dla N = 150 punktów ===
 -> Homologia:  47,110 elementów
 -> Kohomologia:19,503 elementów
 -> Stosunek:   0.41x więcej

=== Zliczanie dla N = 200 punktów ===
 -> Homologia:  111,648 elementów
 -> Kohomologia:64,896 elementów
 -> Stosunek:   0.58x więcej

=== Zliczanie dla N = 250 punktów ===
 -> Homologia:  216,533 elementów
 -> Kohomologia:169,654 elementów
 -> Stosunek:   0.78x więcej

=== Zliczanie dla N = 300 punktów ===
 -> Homologia:  373,661 elementów
 -> Kohomologia:381,555 elementów
 -> Stosunek:   1.02x więcej

=== Zliczanie dla N = 350 punktów ===
 -> Homologia:  587,584 elementów
 -> Kohomologia:715,293 elementów
 -> Stosunek:   1.22x więcej

=== Zliczanie dla N = 400 punktów ===
 -> Homologia:  896,423 elementów
 -> Kohomologia:1,291,434 elementów
 -> Stosune

In [ ]:
import dionysus as d
import numpy as np

def count_elements_in_persistence(points, epsilon):
    # Budujemy kompleks Ripsa
    f = d.fill_rips(points, 3, epsilon)
    
    # 1. Homologia
    m_hom = d.homology_persistence(f)
    hom_elements = sum(len(col) for col in m_hom)
    
    # 2. Kohomologia
    m_coh = d.cohomology_persistence(f, prime=2, keep_cocycles=True)
    coh_elements = sum(len(col.cocycle) for col in m_coh)
    
    return hom_elements, coh_elements

# --- Generowanie danych typu "Alpha Shape-like" ---
# Aby uzyskać rzadszą strukturę, używamy mniejszego epsilona (max_eps)
# i mniejszej liczby punktów na torusie, aby imitować dane de Silvy.
N = 1000 
epsilon = 0.5 # Mniejszy epsilon = rzadszy kompleks = mniej operacji wierszowych

print("Generuję dane...")
# Załóżmy, że masz skrypt ./torus lub użyj prostego generatora:
# (Możesz tu wstawić wywołanie swojego `./torus N 3`)
points = np.random.rand(N, 3) 

print(f"Analiza dla {N} punktów, epsilon={epsilon}...")
hom, coh = count_elements_in_persistence(points, epsilon)

print(f"\nWyniki:")
print(f"Homologia (elementy):   {hom:,}")
print(f"Kohomologia (elementy): {coh:,}")
print(f"Stosunek (coh/hom):     {coh/max(1, hom):.4f}")